<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🚀 LTX-2.3 22B Distilled 1.1 Multiple Subject Reference</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 GPU Edition Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Ingredients-to-Video / Multi-Reference | Wan2GP Engine + Licon MSR LoRA + mmgp Profile 4</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20GPU-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### What is this notebook?

**Multi-Reference Consistent Video Generation (Ingredients-to-Video)** using **LTX-2.3 22B Distilled 1.1 + LiconStudio MSR LoRA**

**GPU:** T4 x2 | **Model:** LTX-2.3 22B Distilled **1.1** Q3_K_M (~14.7GB, streamed via mmgp)
**LoRA:** LiconStudio MSR V1 (~2.7GB, loaded into transformer block)
**Modes:**
- **Background + Subjects (KI):** Provide 2 to 5 images, with the background scene image uploaded first.
- **Subjects Only (I):** Provide 1 to 4 subject/object images.

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**
2. **Turn on Internet** in Settings sidebar
3. Run all cells in order
4. Upload your reference images + prompt in the Gradio UI


---
## Step 1: Environment Setup

Optimizes memory for Kaggle T4 GPU (~30GB RAM, 16GB VRAM).

In [ ]:
import os, gc, psutil

print('=== Kaggle T4 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Environment optimized!')
print('   Kaggle has ~30GB RAM — no swap needed.')

---
## Step 2: Clone Wan2GP & Install Dependencies

In [ ]:
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('GPU Active!')
except Exception:
    print('WARNING: No GPU. Go to Settings → Accelerator → GPU T4 x2')

!git clone https://github.com/DeepBeepMeep/Wan2GP.git

# PyTorch: Kaggle ships 2.4+ which dropped sm_60 (P100) CUDA kernels.
# PyTorch 2.3.1 is the last release with full sm_60 support.
# Install it FIRST, then all other deps.
!pip install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121
# After the cell above runs, click Kernel → Restart & Run All.

!pip install --timeout 120 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 120 --retries 5 -q mmgp gradio gguf soundfile

# --- Programmatic Patch for older ffmpeg compatibility ---
import os
import shutil

video_decode_path = 'Wan2GP/shared/utils/video_decode.py'
if os.path.exists(video_decode_path):
    print(f"Patching {video_decode_path} for older ffmpeg compatibility...")
    with open(video_decode_path, 'r') as f:
        content = f.read()
    
    target_str = 'cmd += ["-fps_mode", "passthrough", "-frames:v", str(requested_frames), "-f", "rawvideo", "-pix_fmt", out_pix_fmt, "pipe:1"]'
    replacement_str = """    # Dynamic check for -fps_mode vs -vsync support
    fps_mode_opt = "-fps_mode"
    try:
        import subprocess
        res = subprocess.run([ffmpeg_path, "-h"], capture_output=True, text=True, errors="ignore")
        if "fps_mode" not in res.stdout and "fps_mode" not in res.stderr:
            fps_mode_opt = "-vsync"
    except Exception:
        pass
    out_pix_fmt = "gbrpf32le" if hdr_linear else "rgb24"
    cmd += [fps_mode_opt, "passthrough", "-frames:v", str(requested_frames), "-f", "rawvideo", "-pix_fmt", out_pix_fmt, "pipe:1"]"""
    
    if target_str in content:
        content = content.replace(target_str, replacement_str)
        with open(video_decode_path, 'w') as f:
            f.write(content)
        print("Successfully patched video_decode.py!")
    else:
        print("Warning: target string not found in video_decode.py!")

plugin_path = 'Wan2GP/plugins/wan2gp-motion-designer/plugin.py'
if os.path.exists(plugin_path):
    print(f"Patching {plugin_path} for older ffmpeg compatibility...")
    with open(plugin_path, 'r') as f:
        content = f.read()
    
    target_plugin = '                    **{\n                        "vsync": "cfr",\n                        "fps_mode": "cfr",\n                        "fflags": "+genpts",\n                        "copyts": None,\n                    },'
    replacement_plugin = '                    **{\n                        "vsync": "cfr",\n                        "fflags": "+genpts",\n                        "copyts": None,\n                    },'
    
    if target_plugin in content:
        content = content.replace(target_plugin, replacement_plugin)
        with open(plugin_path, 'w') as f:
            f.write(content)
        print("Successfully patched plugin.py!")
    elif '"fps_mode": "cfr",' in content:
        content = content.replace('"fps_mode": "cfr",', '')
        with open(plugin_path, 'w') as f:
            f.write(content)
        print("Successfully removed fps_mode from plugin.py (fallback)!")
    else:
        print("Warning: target plugin string not found in plugin.py!")

# --- Programmatic Patches for MSR pipeline adjustments in Wan2GP ---
def apply_programmatic_patches():
    import os
    
    # 1. Patch ltx2.py
    ltx2_py_path = 'Wan2GP/models/ltx2/ltx2.py'
    if os.path.exists(ltx2_py_path):
        print(f"Patching {ltx2_py_path}...")
        with open(ltx2_py_path, 'r') as f:
            content = f.read()
        
        target_str = 'input_video_strength = max(0.0, min(1.0, input_video_strength))'
        replacement_str = 'input_video_strength = max(0.0, min(1.0, input_video_strength)) if input_video_strength is not None else 1.0'
        
        target_msr_str = 'video_conditioning.append((ref_video, 0, control_strength))'
        replacement_msr_str = 'video_conditioning.append((ref_video, -200, control_strength, True))'
        
        # Fallback to catch previously modified index
        target_msr_str_2 = 'video_conditioning.append((ref_video, int(frame_num), control_strength))'
        
        content = content.replace(target_str, replacement_str)
        content = content.replace(target_msr_str, replacement_msr_str)
        content = content.replace(target_msr_str_2, replacement_msr_str)
        
        with open(ltx2_py_path, 'w') as f:
            f.write(content)
        print("Successfully patched ltx2.py")
        
    # 2. Patch keyframe_cond.py
    keyframe_cond_path = 'Wan2GP/models/ltx2/ltx_core/conditioning/types/keyframe_cond.py'
    if os.path.exists(keyframe_cond_path):
        print(f"Patching {keyframe_cond_path}...")
        with open(keyframe_cond_path, 'r') as f:
            content = f.read()
        
        target_init = 'def __init__(self, keyframes: torch.Tensor, frame_idx: int, strength: float):'
        replacement_init = 'def __init__(self, keyframes: torch.Tensor, frame_idx: int, strength: float, keep_all_frames: bool = False):'
        
        target_init_body = '        self.strength = strength'
        replacement_init_body = '        self.strength = strength\n        self.keep_all_frames = keep_all_frames'
        
        target_prepend = '        if self.frame_idx < 0:\n            self.frame_idx  = - self.frame_idx \n            remove_prepend = True'
        replacement_prepend = '        if self.frame_idx < 0 and not getattr(self, "keep_all_frames", False):\n            self.frame_idx  = - self.frame_idx \n            remove_prepend = True'
        
        content = content.replace(target_init, replacement_init)
        content = content.replace(target_init_body, replacement_init_body)
        content = content.replace(target_prepend, replacement_prepend)
        
        with open(keyframe_cond_path, 'w') as f:
            f.write(content)
        print("Successfully patched keyframe_cond.py")

    # 3. Patch reference_video_cond.py
    ref_cond_path = 'Wan2GP/models/ltx2/ltx_core/conditioning/types/reference_video_cond.py'
    if os.path.exists(ref_cond_path):
        print(f"Patching {ref_cond_path}...")
        with open(ref_cond_path, 'r') as f:
            content = f.read()
            
        target_init = 'def __init__(self, latent: torch.Tensor, strength: float = 1.0, frame_idx: int = 0, downscale_factor: int = 1):'
        replacement_init = 'def __init__(self, latent: torch.Tensor, strength: float = 1.0, frame_idx: int = 0, downscale_factor: int = 1, keep_all_frames: bool = False):'
        
        target_init_body = '        self.downscale_factor = max(1, int(downscale_factor))'
        replacement_init_body = '        self.downscale_factor = max(1, int(downscale_factor))\n        self.keep_all_frames = keep_all_frames'
        
        target_prepend = '        remove_prepend = frame_idx < 0'
        replacement_prepend = '        remove_prepend = frame_idx < 0 and not getattr(self, "keep_all_frames", False)'
        
        content = content.replace(target_init, replacement_init)
        content = content.replace(target_init_body, replacement_init_body)
        content = content.replace(target_prepend, replacement_prepend)
        
        with open(ref_cond_path, 'w') as f:
            f.write(content)
        print("Successfully patched reference_video_cond.py")

    # 4. Patch helpers.py
    helpers_path = 'Wan2GP/models/ltx2/ltx_pipelines/utils/helpers.py'
    if os.path.exists(helpers_path):
        print(f"Patching {helpers_path}...")
        with open(helpers_path, 'r') as f:
            content = f.read()
            
        target_keyframe_loop = '''    for entry in video_conditioning:
        if len(entry) == 2:
            video_path, strength = entry
            frame_idx = 0
        elif len(entry) == 3:
            video_path, frame_idx, strength = entry
        else:
            raise ValueError("Video conditioning entries must be (video, strength) or (video, frame_idx, strength).")'''
            
        replacement_keyframe_loop = '''    for entry in video_conditioning:
        keep_all_frames = False
        if len(entry) == 2:
            video_path, strength = entry
            frame_idx = 0
        elif len(entry) == 3:
            video_path, frame_idx, strength = entry
        elif len(entry) == 4:
            video_path, frame_idx, strength, keep_all_frames = entry
        else:
            raise ValueError("Video conditioning entries must be (video, strength), (video, frame_idx, strength), or (video, frame_idx, strength, keep_all_frames).")'''
            
        target_inst1 = '''                VideoConditionByKeyframeIndex(
                        keyframes=encoded_video[:, :, split_latent:],
                        frame_idx=split_frame,
                        strength=strength,
                    )'''
        replacement_inst1 = '''                VideoConditionByKeyframeIndex(
                        keyframes=encoded_video[:, :, split_latent:],
                        frame_idx=split_frame,
                        strength=strength,
                        keep_all_frames=keep_all_frames,
                    )'''
                    
        target_inst2 = '''        cond = VideoConditionByKeyframeIndex(
            keyframes=encoded_video,
            frame_idx=frame_idx,
            strength=strength,
        )'''
        replacement_inst2 = '''        cond = VideoConditionByKeyframeIndex(
            keyframes=encoded_video,
            frame_idx=frame_idx,
            strength=strength,
            keep_all_frames=keep_all_frames,
        )'''
        
        target_ref_loop = '''    for entry in video_conditioning:
        if len(entry) == 2:
            video_path, strength = entry
            frame_idx = 0
        elif len(entry) == 3:
            video_path, frame_idx, strength = entry
        else:
            raise ValueError("Video conditioning entries must be (video, strength) or (video, frame_idx, strength).")'''
            
        replacement_ref_loop = '''    for entry in video_conditioning:
        keep_all_frames = False
        if len(entry) == 2:
            video_path, strength = entry
            frame_idx = 0
        elif len(entry) == 3:
            video_path, frame_idx, strength = entry
        elif len(entry) == 4:
            video_path, frame_idx, strength, keep_all_frames = entry
        else:
            raise ValueError("Video conditioning entries must be (video, strength), (video, frame_idx, strength), or (video, frame_idx, strength, keep_all_frames).")'''
            
        target_ref_inst = '''            VideoConditionByReferenceLatent(
                latent=encoded_video,
                frame_idx=frame_idx,
                strength=strength,
                downscale_factor=scale,
            )'''
        replacement_ref_inst = '''            VideoConditionByReferenceLatent(
                latent=encoded_video,
                frame_idx=frame_idx,
                strength=strength,
                downscale_factor=scale,
                keep_all_frames=keep_all_frames,
            )'''
            
        content = content.replace(target_keyframe_loop, replacement_keyframe_loop)
        content = content.replace(target_inst1, replacement_inst1)
        content = content.replace(target_inst2, replacement_inst2)
        content = content.replace(target_ref_loop, replacement_ref_loop)
        content = content.replace(target_ref_inst, replacement_ref_inst)
        
        with open(helpers_path, 'w') as f:
            f.write(content)
        print("Successfully patched helpers.py")

apply_programmatic_patches()



---
## Step 3: Download All Required Models

Downloads LTX-2.3 22B Distilled 1.1 GGUF model and Gemma-3 text encoder, as well as the specialized **LiconStudio MSR LoRA**.

In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/tmp/hf_home'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

from huggingface_hub import hf_hub_download

REPO = 'Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF'
COMPANION_REPO = 'DeepBeepMeep/LTX-2'
MODEL_DIR = 'Wan2GP/models'
TMP_DIR = '/kaggle/tmp/models'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# Q3_K_M: 14.7 GB — best fit for T4
TRANSFORMER_FILE = 'LTX-2.3-22B-distilled-1.1-Q3_K_M.gguf'
TRANSFORMER_RENAMED = 'ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf'

# Download transformer directly to internal storage (/kaggle/working) to save temp disk space
dest = os.path.join(MODEL_DIR, TRANSFORMER_RENAMED)
if os.path.exists(dest):
    print(f'  ✓ Already exists: {TRANSFORMER_RENAMED}')
else:
    print(f'Downloading {TRANSFORMER_FILE} to {MODEL_DIR} (internal storage)...')
    hf_hub_download(repo_id=REPO, filename=TRANSFORMER_FILE, local_dir=MODEL_DIR, local_dir_use_symlinks=False)
    # Rename it to match the expected name
    actual = os.path.join(MODEL_DIR, TRANSFORMER_FILE)
    if os.path.exists(actual) and actual != dest:
        os.rename(actual, dest)
    print(f'  ✓ {TRANSFORMER_RENAMED} (downloaded directly)')

# Companion large files from DeepBeepMeep - Download to /kaggle/tmp and symlink
LARGE_FILES_COMPANION = [
    'ltx-2.3-22b_embeddings_connector.safetensors',                   # 4.0 GB
    'ltx-2.3-22b_text_embedding_projection.safetensors',              # 2.3 GB
    'ltx-2.3-22b_vae.safetensors',                                    # 1.5 GB
]

for f in LARGE_FILES_COMPANION:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f} → /kaggle/tmp ...')
    hf_hub_download(repo_id=COMPANION_REPO, filename=f, local_dir=TMP_DIR, local_dir_use_symlinks=False)
    actual = os.path.join(TMP_DIR, f)
    os.symlink(actual, dest)
    print(f'  ✓ {f} (symlinked)')

# Download and Symlink the MSR LoRA
os.makedirs(os.path.join(MODEL_DIR, "loras", "ltx2"), exist_ok=True)
dest_lora = os.path.join(MODEL_DIR, "loras", "ltx2", "LTX-2.3-Licon-MSR-V1.safetensors")
if os.path.exists(dest_lora):
    print('  ✓ Already exists: LTX-2.3-Licon-MSR-V1.safetensors')
else:
    print('Downloading LTX-2.3-Licon-MSR-V1.safetensors → /kaggle/tmp ...')
    hf_hub_download(repo_id=COMPANION_REPO, filename="loras/LTX-2.3-Licon-MSR-V1.safetensors", local_dir=TMP_DIR, local_dir_use_symlinks=False)
    actual_lora = os.path.join(TMP_DIR, "loras", "LTX-2.3-Licon-MSR-V1.safetensors")
    os.symlink(actual_lora, dest_lora)
    print('  ✓ LTX-2.3-Licon-MSR-V1.safetensors (symlinked)')

# Small files - Download to /kaggle/tmp and symlink
SMALL_FILES = [
    'ltx-2.3-22b_audio_vae.safetensors',                 # 107 MB
    'ltx-2.3-22b_vocoder.safetensors',                   # 258 MB
    'ltx-2.3-spatial-upscaler-x2-1.1.safetensors',       # 996 MB
    'ltx-2.3-temporal-upscaler-x2-1.0.safetensors',      # 262 MB
]

for f in SMALL_FILES:
    dest = os.path.join(MODEL_DIR, f)
    if os.path.exists(dest):
        print(f'  ✓ Already exists: {f}')
        continue
    print(f'Downloading {f} → /kaggle/tmp ...')
    hf_hub_download(repo_id=COMPANION_REPO, filename=f, local_dir=TMP_DIR, local_dir_use_symlinks=False)
    actual = os.path.join(TMP_DIR, f)
    os.symlink(actual, dest)
    print(f'  ✓ {f} (symlinked)')

# Gemma text encoder → /kaggle/tmp (symlinked)
GEMMA_FOLDER = 'gemma-3-12b-it-qat-q4_0-unquantized'
GEMMA_FILES = [
    'gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors',
    'added_tokens.json', 'chat_template.json', 'config_light.json',
    'generation_config.json', 'preprocessor_config.json', 'processor_config.json',
    'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json',
]

gemma_dest = os.path.join(MODEL_DIR, GEMMA_FOLDER)
gemma_tmp = os.path.join(TMP_DIR, GEMMA_FOLDER)
if os.path.exists(gemma_dest):
    print(f'  ✓ Already exists: {GEMMA_FOLDER}/')
else:
    os.makedirs(gemma_tmp, exist_ok=True)
    for gf in GEMMA_FILES:
        tmp_file = os.path.join(gemma_tmp, gf)
        if os.path.exists(tmp_file): continue
        print(f'Downloading gemma/{gf} → /kaggle/tmp ...')
        hf_hub_download(repo_id=COMPANION_REPO, filename=f'{GEMMA_FOLDER}/{gf}', local_dir=TMP_DIR, local_dir_use_symlinks=False)
    os.symlink(gemma_tmp, gemma_dest)
    print(f'  ✓ {GEMMA_FOLDER}/ (symlinked)')

# Cleanup HF cache
import shutil
for d in [os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache'), '/kaggle/tmp/hf_home']:
    if os.path.exists(d):
        try:
            shutil.rmtree(d)
        except Exception:
            pass

os.system('df -h /kaggle/working /kaggle/tmp')
print('\n✅ All downloads complete!')

---
## Step 4: Write the MSR Generation Script

Creates `run_ltx_msr.py` — pipeline with LTX2 MSR loading, LoRA loading, and Gradio UI.

In [ ]:
%%writefile run_ltx_msr.py
import gc
import os
import sys
import json
import random
import tempfile
import glob
import traceback
import numpy as np
import subprocess
import psutil
import soundfile as sf
from PIL import Image

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

import torch

# Detect GPU architecture once
_GPU_SM = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
_IS_SM60 = (_GPU_SM[0] == 6)   # P100 = sm_60; T4 = sm_75; A100 = sm_80; etc.

if _IS_SM60:
    os.environ["WGP_DTYPE"] = "fp16"
    print(f"  [GPU] sm_60 detected (P100) — FP16 mode + CPU audio patches enabled")
else:
    print(f"  [GPU] sm_{_GPU_SM[0]}{_GPU_SM[1]} detected — native CUDA mode, no patches needed")

import gradio as gr
from shared.utils.audio_video import save_video

def _register_gguf_handler():
    """Register the GGUF handler with mmgp's quant_router."""
    import shared.qtypes.gguf
    print("  [GGUF] ✅ Extension handler registered with mmgp (Wan2GP native)")

def _patch_ltx2_config_loading():
    """Patch _load_config_from_checkpoint to handle GGUF metadata errors."""
    import models.ltx2.ltx2 as ltx2_mod
    _original = ltx2_mod._load_config_from_checkpoint

    def _patched(path, fallback_config_path=None):
        from mmgp import quant_router
        if isinstance(path, (list, tuple)):
            path = path[0] if path else ""
        if not path:
            return {}
        try:
            _, metadata = quant_router.load_metadata_state_dict(path)
            if metadata:
                config_raw = metadata.get("config")
                if config_raw:
                    config = ltx2_mod._normalize_config(config_raw)
                    if config:
                        return config
        except Exception as e:
            print(f"  [GGUF Patch] Metadata read: {type(e).__name__}, using fallback config")
        if fallback_config_path and os.path.isfile(fallback_config_path):
            try:
                with open(fallback_config_path, "r", encoding="utf-8") as f:
                    config = ltx2_mod._normalize_config(json.load(f))
                    if config:
                        print(f"  [GGUF Patch] ✅ Config loaded from {os.path.basename(fallback_config_path)}")
                        return config
            except Exception:
                pass
        return {}

    ltx2_mod._load_config_from_checkpoint = _patched
    print("  [GGUF Patch] ✅ Config loading patched for GGUF")

# ==== GPU INFO ====
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute Capability: {torch.cuda.get_device_capability()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ==== LOAD MODEL & MSR LORA ====
print("\nLoading LTX-2.3 22B Distilled 1.1 MSR Model...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl

fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.ltx2.ltx2_handler import family_handler

_register_gguf_handler()
_patch_ltx2_config_loading()

base_model_type = "ltx2_22B_msr"
model_def = {
    "ltx2_pipeline": "distilled",
    "ltx2_msr": True,
    "ltx2_msr_frame_count": 41
}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = "models/gemma-3-12b-it-qat-q4_0-unquantized"
gemma_files = sorted(glob.glob(os.path.join(gemma_folder, "*.safetensors")))
quanto_files = [f for f in gemma_files if "quanto" in f]
text_encoder_file = quanto_files[0] if quanto_files else (gemma_files[0] if gemma_files else None)
if not text_encoder_file:
    raise FileNotFoundError(f"No .safetensors in {gemma_folder}. Check download cell.")
print(f"  Text encoder: {os.path.basename(text_encoder_file)}")

transformer_path = os.path.join("models", "ltx-2.3-22b-distilled-1.1-Q3_K_M.gguf")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"{transformer_path} missing. Check download cell.")
print(f"  Transformer : {os.path.basename(transformer_path)}")
sys.stdout.flush()

MODEL_DTYPE = torch.float16
VAE_DTYPE   = torch.float16

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type=base_model_type,
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=MODEL_DTYPE,
    VAE_dtype=VAE_DTYPE,
    text_encoder_filename=text_encoder_file,
)

# ==== Apply mmgp Profile 4 ====
print("\nApplying mmgp Profile 4 with budgets...")
sys.stdout.flush()

offload.profile(
    pipe,
    profile_no=4,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    loras=["transformer"],
    budgets={
        "transformer":       6000,
        "text_encoder":      1500,
        "video_encoder":     2000,
        "video_decoder":     3000,
        "audio_encoder":     1000,
        "audio_decoder":     1000,
        "vocoder":           500,
        "spatial_upsampler": 1500,
        "vae":               1000,
        "*":                 1000,
    },
)
offload.shared_state["_attention"] = "sdpa"

# ==== Apply MSR LoRA ====
lora_filename = "LTX-2.3-Licon-MSR-V1.safetensors"
lora_path = os.path.join("models", "loras", "ltx2", lora_filename)
if not os.path.isfile(lora_path):
    lora_path = os.path.join("models", lora_filename)

if os.path.isfile(lora_path):
    print(f"\nApplying MSR LoRA: {lora_path} ...")
    trans_lora, _ = ltx2_model.get_trans_lora()
    preprocess_target = trans_lora
    split_linear_modules_map = getattr(preprocess_target, "split_linear_modules_map", None)
    
    # 8 inference steps mapping
    loras_list_mult_choices_nums = [[1.0] * 8]
    
    def get_loras_preprocessor(transformer, model_type):
        preprocessor = getattr(transformer, "preprocess_loras", None)
        if preprocessor is None:
            return None
        return lambda sd: preprocessor(model_type, sd)
        
    offload.load_loras_into_model(
        trans_lora,
        [lora_path],
        loras_list_mult_choices_nums,
        activate_all_loras=True,
        preprocess_sd=get_loras_preprocessor(preprocess_target, base_model_type),
        pinnedLora=False,
        maxReservedLoras=-1,
        split_linear_modules_map=split_linear_modules_map,
    )
    print("✅ MSR LoRA loaded successfully!")
else:
    print(f"\n⚠️ WARNING: MSR LoRA '{lora_path}' not found!")

# ==== CPU pad workaround for P100 (sm_60) ====
if _IS_SM60:
    try:
        import torch.nn.functional as _F
        from models.ltx2.ltx_core.model.audio_vae.causal_conv_2d import CausalConv2d as _CC2d

        def _cc2d_cpu_pad(self, x: torch.Tensor) -> torch.Tensor:
            if x.is_cuda:
                dev, dt = x.device, x.dtype
                x_cpu = x.detach().cpu().float()
                x_cpu = _F.pad(x_cpu, self.padding)
                w = self.conv.weight.detach().cpu().float()
                b = self.conv.bias.detach().cpu().float() if self.conv.bias is not None else None
                out = _F.conv2d(x_cpu, w, b,
                                self.conv.stride, self.conv.padding,
                                self.conv.dilation, self.conv.groups)
                return out.to(device=dev, dtype=dt)
            else:
                x = _F.pad(x, self.padding)
                return self.conv(x)

        _CC2d.forward = _cc2d_cpu_pad
        print("  [sm_60 Fix] ✅ CausalConv2d patched: pad+conv run on CPU")
    except Exception as _e:
        print(f"  [sm_60 Fix] ❌ Could not patch CausalConv2d: {_e}")

print("\n✅ Setup complete! Distilled MSR Multi-Ref pipeline active.")
sys.stdout.flush()

# ==== HELPER FUNCTIONS ====
def get_resolution(base_res_str, aspect_ratio_str):
    base_resolutions = {"1080p": 1088, "720p": 704, "540p": 544, "480p": 480}
    ratios = {
        "16:9 Landscape": 16/9, "4:3 Standard": 4/3,
        "1:1 Square": 1.0, "3:4 Portrait": 3/4, "9:16 Portrait": 9/16,
    }
    base = base_resolutions.get(base_res_str, 704)
    ratio = ratios.get(aspect_ratio_str, 16/9)
    if ratio >= 1.0:
        height = base
        width = int(base * ratio)
    else:
        width = base
        height = int(base / ratio)
    return (width // 32) * 32, (height // 32) * 32

@torch.inference_mode()
def Video_Generation(prompt, input_ref_images, seed, duration_dropdown,
                     resolution_dropdown, aspect_ratio_dropdown, msr_mode,
                     guide_scale=4.0, num_steps=8, input_audio=None, msr_lora_scale=1.0, progress=gr.Progress()):
    try:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        progress(0, desc="Starting...")

        if not input_ref_images:
            return None, "❌ Error: You must upload at least one reference image."

        ref_images_list = [img.name if hasattr(img, 'name') else img for img in input_ref_images]
        
        # MSR Validation
        if msr_mode == "KI (Background + Subjects)":
            if not (2 <= len(ref_images_list) <= 5):
                return None, "❌ Error: 'Background + Subjects' mode requires 2 to 5 reference images, with the background scene image uploaded FIRST."
            video_prompt_type = "KI"
        else:
            if not (1 <= len(ref_images_list) <= 4):
                return None, "❌ Error: 'Subjects Only' mode requires 1 to 4 reference images."
            video_prompt_type = "I"

        duration_map = {
            "2 Seconds (49 frames)":  49,
            "3 Seconds (73 frames)":  73,
            "5 Seconds (121 frames)": 121,
            "8 Seconds (193 frames)": 193,
            "10 Seconds (241 frames)": 241,
            "15 Seconds (361 frames)": 361,
            "20 Seconds (481 frames)": 481,
            "25 Seconds (601 frames)": 601,
            "30 Seconds (721 frames)": 721,
        }
        frame_rate = 24.0
        num_frames = duration_map.get(duration_dropdown, 121)
        width, height = get_resolution(resolution_dropdown, aspect_ratio_dropdown)

        if seed is None or seed < 0:
            seed = random.randint(0, 2**32 - 1)
        seed = int(seed)

        # Optional audio handling
        input_waveform, input_waveform_sample_rate = None, None
        audio_prompt_type = ""
        if input_audio is not None:
            wav, sr = sf.read(input_audio)
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            input_waveform = wav.astype(np.float32)
            input_waveform_sample_rate = int(sr)
            audio_prompt_type = "A"
            print(f"  [Audio] Loaded soundtrack: sr={sr}, duration={len(wav)/sr:.1f}s")

        # Apply/Update MSR LoRA dynamically with the chosen strength
        if os.path.isfile(lora_path):
            print(f"  [MSR LoRA] Dynamically setting MSR LoRA scale to: {msr_lora_scale} ...")
            trans_lora, _ = ltx2_model.get_trans_lora()
            preprocess_target = trans_lora
            split_linear_modules_map = getattr(preprocess_target, "split_linear_modules_map", None)
            loras_list_mult_choices_nums = [[float(msr_lora_scale)] * 8]
            
            def get_loras_preprocessor(transformer, model_type):
                preprocessor = getattr(transformer, "preprocess_loras", None)
                if preprocessor is None:
                    return None
                return lambda sd: preprocessor(model_type, sd)
                
            offload.load_loras_into_model(
                trans_lora,
                [lora_path],
                loras_list_mult_choices_nums,
                activate_all_loras=True,
                preprocess_sd=get_loras_preprocessor(preprocess_target, base_model_type),
                pinnedLora=False,
                maxReservedLoras=-1,
                split_linear_modules_map=split_linear_modules_map,
            )
            print("  [MSR LoRA] ✅ LoRA loaded/updated successfully!")
        else:
            print(f"  [MSR LoRA] ⚠️ WARNING: MSR LoRA not found at {lora_path}!")

        free_vram = torch.cuda.mem_get_info()[0] / 1024**3
        ram = psutil.virtual_memory()
        print(f"\n{'='*60}")
        print(f"Generating MSR Video: {width}x{height}, {num_frames} frames, seed={seed}")
        print(f"  MSR Mode: {msr_mode} | Reference Count: {len(ref_images_list)}")
        print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
        print(f"  Audio Soundtrack: {'YES' if input_waveform is not None else 'NO'}")
        print(f"  Prompt: {prompt[:120]}{'...' if len(prompt) > 120 else ''}")
        print(f"{'='*60}")
        sys.stdout.flush()

        vae_tile_size = 256
        total_steps = [8]
        current_step = [0]
        current_pass = [1]

        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps is not None:
                    total_steps[0] = override_num_inference_steps
                if pass_no is not None:
                    current_pass[0] = pass_no
                current_step[0] = 0
                return
            current_step[0] += 1
            stage_name = "Stage 1 (half-res)" if current_pass[0] == 1 else ("Stage 2 (full-res refine)" if current_pass[0] == 2 else "Diffusion")
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"  [{stage_name}] step {current_step[0]}/{total_steps[0]} | VRAM free: {free_v:.2f} GB")
            sys.stdout.flush()

            frac = current_step[0] / max(total_steps[0], 1)
            frac = (0.73 + 0.22 * frac) if current_pass[0] == 2 else (frac * 0.73)
            progress(min(frac, 0.95), desc=f"{stage_name}: {current_step[0]}/{total_steps[0]}")

        gen_kwargs = dict(
            input_prompt=prompt,
            input_ref_images=ref_images_list,
            height=height,
            width=width,
            frame_num=num_frames,
            fps=frame_rate,
            seed=seed,
            callback=cb,
            VAE_tile_size=vae_tile_size,
            input_video_strength=1.0,
            denoising_strength=1.0,
            guide_scale=float(guide_scale),
            sampling_steps=int(num_steps),
            guide_phases=2,
            n_prompt="",
            video_prompt_type=video_prompt_type,
            audio_prompt_type=audio_prompt_type,
        )

        if input_waveform is not None:
            gen_kwargs["input_waveform"] = input_waveform
            gen_kwargs["input_waveform_sample_rate"] = input_waveform_sample_rate
            gen_kwargs["audio_scale"] = 1.0

        _stage_labels = {
            "VAE Encoding":  ("🎞️ VAE Encoding reference frames...",       0.05),
            "VAE Decoding":  ("🎬 VAE Decoding latents → frames...",     0.88),
            "Upsampling":    ("🔭 Spatial upsampling latents...",         0.83),
        }

        import time as _time
        _t = [_time.time()]

        def _elapsed():
            now = _time.time()
            dt = now - _t[0]
            _t[0] = now
            return dt

        def set_progress_status(status: str):
            dt = _elapsed()
            label, frac = _stage_labels.get(status, (f"⏳ {status}...", 0.85))
            print(f"  [{status}] {label}  (+{dt:.1f}s)")
            sys.stdout.flush()
            progress(frac, desc=label)

        gen_kwargs["set_progress_status"] = set_progress_status

        print("  Starting generation pipeline...")
        sys.stdout.flush()
        _t[0] = _time.time()

        result = ltx2_model.generate(**gen_kwargs)
        print(f"  [pipeline total] {_time.time() - _t[0]:.1f}s")

        progress(0.97, desc="✅ Saving video...")
        sys.stdout.flush()

        if result is None:
            return None, "Generation failed or was interrupted."

        # Parse output
        audio_data = None
        audio_sr = None
        if isinstance(result, dict):
            video_tensor = result.get("x")
            audio_data = result.get("audio")
            audio_sr = result.get("audio_sampling_rate", 24000)
        else:
            video_tensor = result

        if video_tensor is None or not torch.is_tensor(video_tensor):
            return None, f"❌ No video tensor. Got: {type(video_tensor)}"

        video_tensor = video_tensor.cpu()
        gc.collect(); torch.cuda.empty_cache()

        # Save output video
        out_path = tempfile.mktemp(suffix=".mp4")
        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0
        save_video(tensor=video_for_save, save_file=out_path, fps=frame_rate, normalize=True, value_range=(-1, 1))

        # Audio Muxing
        if input_audio is not None and audio_data is None:
            audio_data = input_waveform
            audio_sr = input_waveform_sample_rate

        if audio_data is not None:
            try:
                audio_tmp = tempfile.mktemp(suffix=".wav")
                if isinstance(audio_data, np.ndarray):
                    audio_np = audio_data
                    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
                        audio_np = audio_np.T
                    sf.write(audio_tmp, audio_np, int(audio_sr or 24000))

                final_path = out_path.replace(".mp4", "_with_audio.mp4")
                subprocess.run([
                    "ffmpeg", "-y", "-i", out_path, "-i", audio_tmp,
                    "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                    "-shortest", final_path
                ], check=True, capture_output=True)
                if os.path.exists(final_path) and os.path.getsize(final_path) > 0:
                    out_path = final_path
            except Exception as e:
                print(f"  ⚠️ Audio mux failed: {e}")

        del video_tensor, video_for_save
        gc.collect(); torch.cuda.empty_cache()
        progress(1.0, desc="Done!")
        return out_path, f"✅ Done! Seed: {seed} | {width}x{height} | {num_frames} frames"

    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        return None, f"❌ Error: {str(e)}"

# ==== GRADIO UI ====
CSS = """@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(56,239,125,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="LTX-2.3 MSR Multi-Ref | AIQUEST") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🌌 LTX-2.3 Distilled 1.1 — Multiple Subject Reference</div><div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> | Kaggle T4 GPU</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        "**Multi-Reference Consistent Video Generation** using LTX-2.3 22B Distilled 1.1 + LiconStudio MSR LoRA.\\n\\n"
        "### 💡 How to use the MSR Composition Modes:\\n"
        "- **Subjects Only (I) [Recommended for character-only reference]:** Upload 1 to 4 subject/character images. The background is generated dynamically from your text prompt (e.g. 'kissing in a garden').\\n"
        "- **Background + Subjects (KI):** Upload 2 to 5 images total. The **FIRST** image in the list must be the background scene image, followed by the subject images. The characters will be composed onto that specific background scene."
    )

    with gr.Column():
        prompt = gr.Textbox(label="📝 Prompt (describe the final scene motion and interaction)", lines=3,
                   placeholder="A tracking shot of the character in the kitchen, preparing dinner, high quality cinematic look.")

        with gr.Row():
            input_ref_images = gr.Files(file_count="multiple", label="🖼️ MSR Reference Images (Drag/Upload multiple files)")
            msr_mode = gr.Dropdown(
                label="⚙️ MSR Composition Mode",
                choices=["KI (Background + Subjects)", "I (Subjects / Objects Only)"],
                value="KI (Background + Subjects)"
            )

        with gr.Row():
            input_audio = gr.Audio(type="filepath", label="🎵 Optional Soundtrack Audio (drifts Lip Sync / timing)")
            seed = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)

        with gr.Row():
            duration_dropdown = gr.Dropdown(
                label="⏱️ Duration",
                choices=[
                    "2 Seconds (49 frames)",
                    "3 Seconds (73 frames)",
                    "5 Seconds (121 frames)",
                    "8 Seconds (193 frames)",
                    "10 Seconds (241 frames)",
                    "15 Seconds (361 frames)",
                    "20 Seconds (481 frames)",
                    "25 Seconds (601 frames)",
                    "30 Seconds (721 frames)",
                ],
                value="5 Seconds (121 frames)",
            )
            resolution_dropdown = gr.Dropdown(
                label="📐 Base Resolution",
                choices=["1080p", "720p", "540p", "480p"],
                value="720p",
            )
            aspect_ratio_dropdown = gr.Dropdown(
                label="📏 Aspect Ratio",
                choices=["16:9 Landscape", "4:3 Standard", "1:1 Square", "3:4 Portrait", "9:16 Portrait"],
                value="16:9 Landscape",
            )

        with gr.Row():
            guide_scale = gr.Slider(
                label="🎯 Prompt Influence (guide_scale)",
                minimum=1.0, maximum=8.0, step=0.5, value=4.0,
            )
            num_steps = gr.Slider(
                label="⚡ Diffusion Steps (8 is optimal for distilled)",
                minimum=2, maximum=8, step=1, value=8,
            )
            msr_lora_scale = gr.Slider(
                label="🧬 MSR LoRA Strength (Character consistency)",
                minimum=0.5, maximum=2.0, step=0.1, value=1.0,
            )

        with gr.Row():
            gen_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg", elem_id="gen-btn")
            stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
            clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")
            
        video_out = gr.Video(label="🎥 Output Video")
        status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

        gen_event = gen_btn.click(
            fn=Video_Generation,
            inputs=[prompt, input_ref_images, seed, duration_dropdown,
                    resolution_dropdown, aspect_ratio_dropdown, msr_mode,
                    guide_scale, num_steps, input_audio, msr_lora_scale],
            outputs=[video_out, status_out],
        )
        stop_btn.click(fn=None, cancels=[gen_event])
        clear_btn.click(
            fn=lambda: (None, None, "", -1),
            outputs=[video_out, status_out, prompt, seed],
        )

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🌌 Created by <strong>AIQuest Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free & Open Source | LTX-2.3 22B Distilled | Kaggle T4 GPU</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #11998e; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #11998e; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #11998e; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

print("\nLaunching Gradio...")
sys.stdout.flush()
demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, max_threads=1, ssr_mode=False)

---
## Step 4b: AetherStudio API layer

The upstream notebook ships only a Gradio UI, which AetherStudio cannot call. This cell rewrites
the launch tail of `run_ltx_msr.py` into a FastAPI service:

| Endpoint | Purpose |
|---|---|
| `GET /v1/health` | Capability probe - modes, ref limits, durations, resolutions |
| `POST /generate` | Scene generation from reference images |
| `GET /outputs/{id}.mp4` | Rendered clip |
| `/gradio` | The original UI, still there for manual tests |

Two things are handled server-side because ffmpeg lives here, not in the browser:

- **Generate-longer-then-trim.** LTX only emits 2/3/5/8/10/15/20/25/30s (`frames = 24*sec + 1`).
  `target_sec` picks the next size up and trims to the exact narration length, so clips stay
  frame-accurate against the voice track.
- **Crop to standard height.** `get_resolution()` floors to multiples of 32, so 1080p/16:9 renders
  **1920x1088**. Pass `output_height: 1080` for a centre crop.

Set `NGROK_TOKEN` before running. The patch aborts loudly if the upstream code has shifted,
instead of silently doing nothing.


In [ ]:
# ============================================================
#  AetherStudio API layer for LTX-2.3 MSR
#  Rewrites the Gradio launch tail of run_ltx_msr.py into FastAPI.
# ============================================================

NGROK_TOKEN = ""   # <-- paste your ngrok authtoken here

!pip install -q fastapi "uvicorn[standard]" pyngrok python-multipart

import os

PATH = "run_ltx_msr.py"
src = open(PATH, encoding="utf-8").read()


def must_replace(text, old, new, label):
    """Fail loudly. A silent no-op .replace() cost us a whole debugging session
    on the Fish Speech notebook — never again."""
    if old not in text:
        raise RuntimeError(
            "PATCH FAILED [" + label + "]: target text not found in " + PATH + ".\n"
            "The upstream notebook changed. Fix this patch before running — do NOT ignore this."
        )
    return text.replace(old, new, 1)


OLD_TAIL = r'''print("\nLaunching Gradio...")
sys.stdout.flush()
demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True, max_threads=1, ssr_mode=False)'''

NEW_TAIL = r'''# ==================== AetherStudio API ====================
import base64, io, uuid, threading, time
import subprocess as _sp
from typing import List, Optional

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
import uvicorn

OUT_DIR = "/kaggle/working/aether_outputs"
REF_DIR = "/kaggle/working/aether_refs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REF_DIR, exist_ok=True)

# LTX emits only these lengths. frames = 24 * sec + 1 at 24fps.
DURATIONS = [2, 3, 5, 8, 10, 15, 20, 25, 30]
DURATION_LABELS = {
    2:  "2 Seconds (49 frames)",
    3:  "3 Seconds (73 frames)",
    5:  "5 Seconds (121 frames)",
    8:  "8 Seconds (193 frames)",
    10: "10 Seconds (241 frames)",
    15: "15 Seconds (361 frames)",
    20: "20 Seconds (481 frames)",
    25: "25 Seconds (601 frames)",
    30: "30 Seconds (721 frames)",
}


class NoProgress:
    """Video_Generation defaults progress to gr.Progress(), which needs a live
    Gradio request context. Supply an inert stand-in for API calls."""

    def __call__(self, frac=0, desc=""):
        return None

    def tqdm(self, iterable, *a, **k):
        return iterable


# Whether this build can render from text alone.
#
# The upstream notebook loads the MSR (reference-conditioned) model, so its own
# generation function refuses a request with no images. The base LTX-2.3 weights
# underneath do support text-to-video, and the MSR LoRA is additive — so calling
# the pipeline directly with video_prompt_type="" and no references SHOULD work.
#
# "Should" is not good enough to advertise as a capability, so this starts as
# "untested" and only becomes True/False after POST /selftest/t2v actually runs a
# tiny generation. /v1/health reports the measured value, never an assumption.
T2V_STATUS = "untested"
T2V_ERROR = ""


class GenReq(BaseModel):
    prompt: str
    images: List[str] = []            # base64; background FIRST when mode == "KI"
    mode: str = "I"                   # "I" (subjects only), "KI" (background + subjects), "T2V" (no references)
    seed: int = -1
    target_sec: float = 5.0           # exact length wanted; we render >= this and trim
    resolution: str = "720p"
    aspect: str = "16:9 Landscape"
    guide_scale: float = 4.0
    steps: int = 8
    msr_lora_scale: float = 1.0
    output_height: Optional[int] = None   # centre-crop, e.g. 1088 -> 1080
    keep_audio: bool = False              # LTX can emit its own audio; off by default
    return_b64: bool = False


app = FastAPI(title="AetherStudio LTX-2.3 MSR")
app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)


def _pick_duration(target):
    for d in DURATIONS:
        if d >= target - 0.001:
            return d
    return DURATIONS[-1]


def _health():
    return {
        "status": "ok",
        "model": "LTX-2.3-22B-distilled-1.1-Q3_K_M",
        "lora": "LiconStudio-MSR-V1",
        "modes": ["I", "KI", "T2V"],
        "max_refs": {"I": 4, "KI": 5, "T2V": 0},
        "durations_sec": DURATIONS,
        "fps": 24,
        "resolutions": ["1080p", "720p", "540p", "480p"],
        "aspects": ["16:9 Landscape", "4:3 Standard", "1:1 Square",
                    "3:4 Portrait", "9:16 Portrait"],
        # "untested" | True | False — measured by /selftest/t2v, never assumed.
        "supports_text_only": T2V_STATUS,
        "text_only_error": T2V_ERROR,
        "supports_negative_prompt": False,
    }


@torch.inference_mode()
def _generate_t2v(prompt, seconds, resolution, aspect, seed, guide_scale, steps,
                  msr_lora_scale=0.0):
    """Text-to-video: drive the pipeline directly, with no reference conditioning.

    Bypasses Video_Generation entirely because that function's first act is to
    reject an empty reference list. Everything else mirrors it, except
    video_prompt_type is "" (no K, no I) so no conditioning is attached.

    msr_lora_scale defaults to 0 here: the MSR LoRA exists to bind output to
    reference images, and with no references it can only distort a text-driven
    scene.
    """
    gc.collect()
    torch.cuda.empty_cache()

    width, height = get_resolution(resolution, aspect)
    num_frames = 24 * int(seconds) + 1

    if os.path.isfile(lora_path):
        trans_lora, _ = ltx2_model.get_trans_lora()
        split_map = getattr(trans_lora, "split_linear_modules_map", None)

        def _pp(transformer, model_type):
            p = getattr(transformer, "preprocess_loras", None)
            return None if p is None else (lambda sd: p(model_type, sd))

        offload.load_loras_into_model(
            trans_lora, [lora_path], [[float(msr_lora_scale)] * 8],
            activate_all_loras=True,
            preprocess_sd=_pp(trans_lora, base_model_type),
            pinnedLora=False, maxReservedLoras=-1,
            split_linear_modules_map=split_map,
        )

    result = ltx2_model.generate(
        input_prompt=prompt,
        input_ref_images=[],
        height=height,
        width=width,
        frame_num=num_frames,
        fps=24.0,
        seed=int(seed),
        callback=None,
        VAE_tile_size=256,
        input_video_strength=1.0,
        denoising_strength=1.0,
        guide_scale=float(guide_scale),
        sampling_steps=int(steps),
        guide_phases=2,
        n_prompt="",
        video_prompt_type="",     # <- no conditioning: pure text-to-video
        audio_prompt_type="",
    )
    if result is None:
        raise RuntimeError("pipeline returned nothing")

    video_tensor = result.get("x") if isinstance(result, dict) else result
    if video_tensor is None or not torch.is_tensor(video_tensor):
        raise RuntimeError("no video tensor returned (got %s)" % type(video_tensor))

    out_path = tempfile.mktemp(suffix=".mp4")
    save_video(
        tensor=video_tensor.cpu().unsqueeze(0).float() / 127.5 - 1.0,
        save_file=out_path, fps=24.0, normalize=True, value_range=(-1, 1),
    )
    del video_tensor
    gc.collect()
    torch.cuda.empty_cache()
    return out_path


@app.post("/selftest/t2v")
def selftest_t2v():
    """Find out for real whether this build can do text-to-video.

    Renders the smallest possible clip (2s at 480p). Takes a minute or two, and
    is the difference between reporting a measured capability and guessing."""
    global T2V_STATUS, T2V_ERROR
    try:
        p = _generate_t2v(
            "a wide shot of an empty field at dawn, slow drifting clouds",
            2, "480p", "16:9 Landscape", 1234, 4.0, 8,
        )
        ok = os.path.isfile(p) and os.path.getsize(p) > 1024
        T2V_STATUS = bool(ok)
        T2V_ERROR = "" if ok else "produced an empty file"
    except Exception as e:
        traceback.print_exc()
        T2V_STATUS = False
        T2V_ERROR = f"{type(e).__name__}: {e}"
    return {"supports_text_only": T2V_STATUS, "error": T2V_ERROR}


@app.get("/v1/health")
def health_v1():
    return _health()


@app.get("/api/health")
def health_api():
    return _health()


# ---- job queue -----------------------------------------------------------
#
# Renders CANNOT be synchronous HTTP requests. Measured against the live
# tunnel: a 269s render returned fine, a 301s render and a 302s render both
# died with ERR_NGROK_3004. ngrok cuts a single request at ~300 seconds, and
# every useful render here takes longer than that — even 3s at 480p.
#
# So /generate now hands back a job id immediately and the work continues in a
# background thread. This is the right shape regardless of the tunnel: a
# twenty-minute blocking request loses real GPU time to any network blip, and
# polling gives the UI genuine progress instead of a spinner.
JOBS = {}
JOBS_LOCK = threading.Lock()
# One render at a time. The GPU is single, mmgp streams weights, and uvicorn
# was launched with max_threads=1 — concurrent renders would OOM, not overlap.
GPU_LOCK = threading.Lock()


def _job_set(job_id, **fields):
    with JOBS_LOCK:
        if job_id in JOBS:
            JOBS[job_id].update(fields)


def _new_job(kind):
    job_id = uuid.uuid4().hex[:12]
    with JOBS_LOCK:
        JOBS[job_id] = {
            "job": job_id,
            "kind": kind,
            "status": "queued",
            "step": 0,
            "total_steps": 0,
            "queued_at": time.time(),
            "started_at": None,
            "finished_at": None,
            "error": "",
        }
    return job_id


def _prune_jobs(keep=40):
    """Keep the map from growing without bound over a long session."""
    with JOBS_LOCK:
        if len(JOBS) <= keep:
            return
        done = [(v.get("finished_at") or 0, k) for k, v in JOBS.items()
                if v.get("status") in ("done", "error")]
        done.sort()
        for _, k in done[:max(0, len(JOBS) - keep)]:
            JOBS.pop(k, None)


def _finish(job, raw_path, req, seed, mode, gen_sec, status="ok"):
    """Trim to the exact narration length, optionally crop, and publish.

    Shared by the reference and text-to-video paths so both honour the
    generate-longer-then-trim contract identically. LTX's own audio is dropped
    unless explicitly wanted, because narration is mixed in AetherStudio.
    """
    final = os.path.join(OUT_DIR, job + ".mp4")
    cmd = ["ffmpeg", "-y", "-i", raw_path, "-t", str(round(float(req.target_sec), 3))]
    if req.output_height:
        h = int(req.output_height)
        cmd += ["-vf", "crop=iw:" + str(h) + ":0:(ih-" + str(h) + ")/2"]
    cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p"]
    cmd += (["-c:a", "aac"] if req.keep_audio else ["-an"])
    cmd += [final]
    _sp.run(cmd, check=True, capture_output=True)

    resp = {
        "job": job,
        "video_url": "/outputs/" + job + ".mp4",
        "seed": seed,
        "mode": mode,
        "generated_sec": gen_sec,
        "final_sec": round(float(req.target_sec), 3),
        "frames": 24 * gen_sec + 1,
        "status": status,
    }
    if req.return_b64:
        with open(final, "rb") as fh:
            resp["video_b64"] = base64.b64encode(fh.read()).decode()
    return resp


@app.post("/generate")
def generate(req: GenReq):
    """Enqueue a render and return immediately. Poll GET /jobs/{id}."""
    _prune_jobs()
    job_id = _new_job("generate")
    t = threading.Thread(target=_render_job, args=(job_id, req), daemon=True)
    t.start()
    return {"job": job_id, "status": "queued", "poll": "/jobs/" + job_id}


@app.get("/jobs/{job_id}")
def job_status(job_id: str):
    with JOBS_LOCK:
        rec = JOBS.get(job_id)
        if not rec:
            return JSONResponse({"error": "unknown job"}, status_code=404)
        out = dict(rec)
    # Elapsed is useful for the UI's "this takes minutes" messaging.
    if out.get("started_at"):
        end = out.get("finished_at") or time.time()
        out["elapsed_sec"] = round(end - out["started_at"], 1)
    return out


@app.get("/jobs")
def jobs_list():
    with JOBS_LOCK:
        return {"jobs": [
            {k: v for k, v in rec.items() if k in ("job", "status", "step", "total_steps", "error")}
            for rec in JOBS.values()
        ]}


def _render_job(job, req: GenReq):
    ref_paths = []
    _job_set(job, status="running", started_at=time.time())
    try:
        mode = req.mode.upper()
        if mode not in ("KI", "I", "T2V"):
            mode = "I"

        # Text-to-video: no references at all. This is the path the storyboard
        # uses for content scenes; reference modes are for the channel host.
        if mode == "T2V":
            seed = int(req.seed)
            if seed < 0:
                seed = random.randint(0, 2**32 - 1)
            gen_sec = _pick_duration(req.target_sec)
            _job_set(job, mode=mode, seed=seed, generated_sec=gen_sec)
            with GPU_LOCK:
                raw = _generate_t2v(
                    req.prompt, gen_sec, req.resolution, req.aspect, seed,
                    req.guide_scale, req.steps, msr_lora_scale=0.0,
                )
            _job_set(job, status="done", finished_at=time.time(),
                     **_finish(job, raw, req, seed, mode, gen_sec))
            return

        if not req.images:
            raise ValueError("modes I and KI need at least one reference image; "
                             "use mode 'T2V' to render from text alone.")

        limit = 5 if mode == "KI" else 4
        if len(req.images) > limit:
            raise ValueError("mode " + mode + " accepts at most " + str(limit) +
                             " images, got " + str(len(req.images)))
        if mode == "KI" and len(req.images) < 2:
            raise ValueError("mode KI needs the background image plus at least one subject.")

        for i, b64 in enumerate(req.images):
            raw = b64.split(",", 1)[1] if "," in b64[:64] else b64
            p = os.path.join(REF_DIR, job + "_" + str(i) + ".png")
            Image.open(io.BytesIO(base64.b64decode(raw))).convert("RGB").save(p)
            ref_paths.append(p)

        # Resolve the seed here rather than inside Video_Generation, so the
        # response can report exactly what produced this clip. Character
        # consistency depends on being able to replay it.
        seed = int(req.seed)
        if seed < 0:
            seed = random.randint(0, 2**32 - 1)

        gen_sec = _pick_duration(req.target_sec)
        mode_label = ("KI (Background + Subjects)" if mode == "KI"
                      else "I (Subjects / Objects Only)")

        _job_set(job, mode=mode, seed=seed, generated_sec=gen_sec)
        with GPU_LOCK:
            out_path, status = Video_Generation(
                req.prompt,
                ref_paths,
                seed,
                DURATION_LABELS[gen_sec],
                req.resolution,
                req.aspect,
                mode_label,
                req.guide_scale,
                req.steps,
                None,
                req.msr_lora_scale,
                progress=NoProgress(),
            )
        if not out_path:
            raise RuntimeError(status or "generation returned nothing")

        _job_set(job, status="done", finished_at=time.time(),
                 **_finish(job, out_path, req, seed, mode, gen_sec, status))

    except Exception as e:
        traceback.print_exc()
        _job_set(job, status="error", finished_at=time.time(),
                 error=f"{type(e).__name__}: {e}")
    finally:
        for p in ref_paths:
            try:
                os.remove(p)
            except Exception:
                pass


app.mount("/outputs", StaticFiles(directory=OUT_DIR), name="outputs")
app = gr.mount_gradio_app(app, demo.queue(), path="/gradio")

print("\nStarting AetherStudio LTX API on :7860 ...")
sys.stdout.flush()

_NGROK_TOKEN = "NGROK_TOKEN_PLACEHOLDER"
if _NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = _NGROK_TOKEN
    _tunnel = ngrok.connect(7860, "http")
    print("=" * 72)
    print("  AETHER LTX ENDPOINT : " + _tunnel.public_url)
    print("  Gradio UI           : " + _tunnel.public_url + "/gradio")
    print("  Paste the endpoint into AetherStudio -> AI Settings -> LTX Video")
    print("=" * 72)
else:
    print("  NGROK_TOKEN is empty - serving on :7860 only (no public URL).")
sys.stdout.flush()

uvicorn.run(app, host="0.0.0.0", port=7860, log_level="info")'''

src = must_replace(src, OLD_TAIL, NEW_TAIL, "gradio launch tail -> AetherStudio FastAPI")
src = src.replace("NGROK_TOKEN_PLACEHOLDER", NGROK_TOKEN)

open(PATH, "w", encoding="utf-8").write(src)
print("Patched run_ltx_msr.py -> FastAPI + ngrok (Gradio still at /gradio)")
if not NGROK_TOKEN:
    print("WARNING: NGROK_TOKEN is empty. Set it above or you get no public URL.")


---
## Step 5: Launch! 🚀

Runs the generation script. Watch for the **public Gradio URL** in the output.

In [ ]:
!cd /kaggle/working && python -u run_ltx_msr.py 2>&1

---

<div align="center">

### 🎉 Enjoyed this notebook?

If this was helpful, please **⬆️ upvote** and **subscribe** for more free AI tools & tutorials!

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>
<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>